# Kaggle Resume: SmolLM2 Kuzu GraphRAG QLoRA

Resume the paused 5,000-row, r=16 QLoRA diagnostic run from optimizer step 400. This is a training driver, not a quality gate: its mix contains graph extraction and Neo4j Text2Cypher examples, not the complete Kuzu/GraphRAG task suite.

Before running, attach the private Kaggle dataset described in `docs/KAGGLE_RESUME.md`, select a GPU, and enable Internet so the pinned code and base model can be fetched. Run cells from the top. The notebook imposes a 10-hour total budget, pauses at an optimizer step with a complete checkpoint, and places resumable output under `/kaggle/working/kaggle_resume_bundle`.

Training loss and validation loss are diagnostics only; they do not establish semantic extraction or Cypher correctness.

In [ ]:
from pathlib import Path
import hashlib
import json
import shutil
import subprocess
import sys
import time

SESSION_STARTED = time.monotonic()
SESSION_BUDGET_SECONDS = 10 * 60 * 60
FINALIZATION_RESERVE_SECONDS = 15 * 60  # reserve for final checkpoint, ZIP verification, and notebook save
INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")
RUN_DIR = WORK_ROOT / "kg_llm_tune_run"
EXPORT_DIR = WORK_ROOT / "kaggle_resume_bundle"
REPO_DIR = WORK_ROOT / "KG-LLM-TUNE"
SOURCE_COMMIT = "261fdf5"
MODEL_ID = "HuggingFaceTB/SmolLM2-360M-Instruct"
MODEL_REVISION = "a10cc1512eabd3dde888204e902eca88bddb4951"

print(f"10-hour session budget started; working directory: {WORK_ROOT}")

## Validate uploaded data and resume state

The input package must contain `train.jsonl`, `val.jsonl`, `manifest.json`, and a run bundle containing `config.yaml`, `env.json`, `metrics.json`, `log.txt`, plus a complete checkpoint (`state.pt`, manifest, and adapter config). Hashes and checkpoint signature are checked before training.

In [ ]:
from zipfile import ZipFile


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


data_candidates = []
for train_path in INPUT_ROOT.rglob("train.jsonl"):
    candidate = train_path.parent
    val_path = candidate / "val.jsonl"
    manifest_path = candidate / "manifest.json"
    if val_path.is_file() and manifest_path.is_file():
        try:
            manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
            if manifest.get("train_sha256") and manifest.get("val_sha256"):
                data_candidates.append((candidate, manifest))
        except (OSError, json.JSONDecodeError):
            pass

if not data_candidates:
    raise FileNotFoundError("Could not find the prepared train.jsonl, val.jsonl, and manifest.json under /kaggle/input.")
DATA_DIR, DATA_MANIFEST = data_candidates[0]
TRAIN_PATH = DATA_DIR / "train.jsonl"
VAL_PATH = DATA_DIR / "val.jsonl"
assert sha256_file(TRAIN_PATH) == DATA_MANIFEST["train_sha256"], "Training data hash mismatch"
assert sha256_file(VAL_PATH) == DATA_MANIFEST["val_sha256"], "Validation data hash mismatch"

resume_archives = list(INPUT_ROOT.rglob("kg-llm-tune-resume-*.zip"))
EXTRACT_ROOTS = []
for resume_archive in resume_archives:
    extract_root = WORK_ROOT / f"uploaded_{resume_archive.stem}"
    if not extract_root.exists():
        extract_root.mkdir(parents=True)
        with ZipFile(resume_archive) as archive:
            root = extract_root.resolve()
            for member in archive.infolist():
                (extract_root / member.filename).resolve().relative_to(root)
            archive.extractall(extract_root)
    EXTRACT_ROOTS.append(extract_root)

complete_checkpoints = []
search_roots = [INPUT_ROOT, *EXTRACT_ROOTS]
if RUN_DIR.exists():
    search_roots.append(RUN_DIR)
for search_root in search_roots:
    for manifest_path in search_root.rglob("checkpoints/step-*/manifest.json"):
        checkpoint_dir = manifest_path.parent
        state_path = checkpoint_dir / "state.pt"
        adapter_config = checkpoint_dir / "adapter" / "adapter_config.json"
        if not (state_path.is_file() and adapter_config.is_file()):
            continue
        checkpoint_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        signature = checkpoint_manifest["signature"]
        if signature.get("train_sha256") != DATA_MANIFEST["train_sha256"] or signature.get("val_sha256") != DATA_MANIFEST["val_sha256"]:
            continue
        run_bundle = checkpoint_dir.parent.parent
        if all((run_bundle / name).is_file() for name in ("config.yaml", "env.json", "metrics.json", "log.txt")):
            complete_checkpoints.append((checkpoint_manifest["optimizer_steps"], checkpoint_dir, run_bundle))

if not complete_checkpoints:
    raise FileNotFoundError("No complete checkpoint matching this dataset was found. Attach the original upload package or a prior Kaggle resume output.")

optimizer_step, CHECKPOINT_DIR, SOURCE_RUN_DIR = max(complete_checkpoints, key=lambda item: item[0])
if SOURCE_RUN_DIR.resolve() != RUN_DIR.resolve():
    if RUN_DIR.exists():
        shutil.rmtree(RUN_DIR)
    shutil.copytree(SOURCE_RUN_DIR, RUN_DIR)
    CHECKPOINT_DIR = RUN_DIR / "checkpoints" / CHECKPOINT_DIR.name

print({"train_rows": DATA_MANIFEST["train_examples"], "validation_rows": DATA_MANIFEST["val_examples"],
       "checkpoint_step": optimizer_step, "checkpoint": str(CHECKPOINT_DIR),
       "dataset_dir": str(DATA_DIR)})

## Prepare the Kaggle GPU runtime

Kaggle supplies the CUDA-enabled PyTorch runtime. The install below pins the project’s model-training dependencies without replacing Torch. The base model is downloaded at its pinned Hugging Face revision during trainer startup; do not upload base-model weights.

In [ ]:
packages = [
    "transformers==5.17.0", "peft==0.21.0", "accelerate==1.15.0",
    "bitsandbytes==0.50.2", "numpy==2.4.6", "PyYAML==6.0.3",
]
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade-strategy", "only-if-needed", *packages], check=True)

import torch
if not torch.cuda.is_available():
    raise RuntimeError("A Kaggle GPU accelerator is required. Select a GPU and rerun from the top.")
if torch.cuda.device_count() != 2:
    raise RuntimeError(f"This recipe expects both T4 GPUs; detected {torch.cuda.device_count()} visible GPU(s).")
print({"gpus": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())], "torch": torch.__version__, "cuda": torch.version.cuda})

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", "open-data-docred-pilot"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", SOURCE_COMMIT], check=True)
else:
    subprocess.run(["git", "clone", "https://github.com/Kartik-A-1820/KG-LLM-TUNE.git", str(REPO_DIR)], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", SOURCE_COMMIT], check=True)

assert (REPO_DIR / "scripts" / "train_lora_smoke.py").is_file()
print("Trainer source:", SOURCE_COMMIT)

## Resume and monitor

The run resumes at the next saved batch with its optimizer and RNG states. This recipe uses both Kaggle T4 GPUs with DDP, one 4k-token microbatch per GPU, and gradient accumulation 4 (global effective batch 8, matching the prior single-GPU recipe). It saves complete checkpoints every 100 optimizer steps and also saves one when the 10-hour budget expires.

In [ ]:
elapsed = time.monotonic() - SESSION_STARTED
max_run_seconds = int(SESSION_BUDGET_SECONDS - elapsed - FINALIZATION_RESERVE_SECONDS)
if max_run_seconds < 300:
    raise RuntimeError("Less than five minutes remain in the 10-hour budget; stop and restart the notebook session before training.")

command = [
    sys.executable, "-m", "torch.distributed.run", "--standalone", "--nproc_per_node=2",
    str(REPO_DIR / "scripts" / "train_lora_smoke.py"),
    "--model", MODEL_ID, "--model-revision", MODEL_REVISION,
    "--train-jsonl", str(TRAIN_PATH), "--val-jsonl", str(VAL_PATH),
    "--source-dataset", "docred-distant+neo4j-text2cypher-2024v1",
    "--source-license", "MIT-and-Apache-2.0-card-component-review-open",
    "--source-manifest", str(DATA_DIR / "manifest.json"),
    "--output-dir", str(RUN_DIR), "--epochs", "3",
    "--max-train-examples", "5000", "--max-val-examples", "500",
    "--max-length", "4096", "--batch-size", "1", "--grad-accum-steps", "4",
    "--lr", "0.0002", "--seed", "1820", "--lora-r", "16", "--lora-alpha", "32",
    "--lora-dropout", "0.05", "--qlora", "--bnb-4bit-quant-type", "nf4",
    "--bnb-4bit-use-double-quant", "--attn-implementation", "sdpa", "--chat-template",
    "--checkpoint-every-steps", "100", "--keep-last-checkpoints", "2",
    "--log-every-steps", "10", "--resume-from", str(CHECKPOINT_DIR),
    "--allow-world-size-migration",
    "--max-run-seconds", str(max_run_seconds), "--save-adapter",
]

stdout_path = RUN_DIR / "kaggle_stdout.log"
with stdout_path.open("w", encoding="utf-8") as stdout_file:
    process = subprocess.Popen(command, cwd=REPO_DIR, stdout=stdout_file, stderr=subprocess.STDOUT)
    last_report = time.monotonic()
    while process.poll() is None:
        time.sleep(60)
        now = time.monotonic()
        if now - last_report >= 300:
            log_path = RUN_DIR / "log.txt"
            tail = log_path.read_text(encoding="utf-8", errors="replace").splitlines()[-3:] if log_path.exists() else []
            print("elapsed_minutes=", round((now - SESSION_STARTED) / 60, 1), "recent_log=", tail)
            last_report = now

training_exit_code = process.returncode
if training_exit_code != 0:
    print(stdout_path.read_text(encoding="utf-8", errors="replace")[-8000:])

metrics = json.loads((RUN_DIR / "metrics.json").read_text(encoding="utf-8"))
print({"status": metrics.get("status"), "optimizer_steps": metrics.get("optimizer_steps"),
       "stop_reason": metrics.get("stop_reason"), "elapsed_seconds": metrics.get("elapsed_seconds")})

## Export one verified resume ZIP

This cell writes `/kaggle/working/kg-llm-tune-resume-step-NNNNNNNN.zip`. It includes the adapter, optimizer/scaler/RNG and data-position state, checkpoint manifests, run config, environment, metrics, and logs. The ZIP is CRC- and SHA-256-verified before use. Download this single ZIP and add it to a new private Kaggle Dataset for the next 10-hour session; keep the original train/validation dataset attached too. Kaggle Save Version also retains the ZIP in notebook output.

In [ ]:
sys.path.insert(0, str(REPO_DIR / "src"))
from kg_llm_tune.resume_bundle import build_resume_archive
metrics = json.loads((RUN_DIR / "metrics.json").read_text(encoding="utf-8"))
optimizer_step = int(metrics["optimizer_steps"])
archive_path = WORK_ROOT / f"kg-llm-tune-resume-step-{optimizer_step:08d}.zip"
archive_report = build_resume_archive(RUN_DIR, archive_path, SOURCE_COMMIT, MODEL_ID)
print(archive_report)
print("The verified ZIP is the continuity artifact. Save a Kaggle notebook version to retain it in notebook output; upload the ZIP as a new private dataset for the next session.")
if training_exit_code != 0:
    raise subprocess.CalledProcessError(training_exit_code, command)